# Phase 4 - Customer Retention Modeling (Spark ML)
## Predicting 90-Day Repeat Purchases on Olist E-Commerce Data

This notebook trains and evaluates distributed machine learning models to predict whether a first-time buyer will make a repeat purchase within 90 days (`did_purchase_again = 1`).

In [ ]:
# Create and validate our Spark session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("projet-final") \
        .master("local[*]") \
        .getOrCreate()

print("Spark Version:", spark.version)

In [ ]:
# Load feature engineered dataset from HDFS
hdfs_path = "hdfs://namenode:9000/user/mti850/analytics/first_orders_features.parquet"
df = spark.read.parquet(hdfs_path)

## 1. Class Imbalance Context

In our first-order cohort, the target variable distribution is highly imbalanced:

* **Class 0 (Did not repurchase within 90 days)**: 97.7% (~73,543 customers)
* **Class 1 (Repurchased within 90 days)**: 2.3% (~1,714 customers)

A naive, "lazy" classifier predicting class 0 for every customer would achieve **97.7% accuracy** without learning any meaningful pattern. However, it would detect 0% of the actual repeat customers (0% recall). Therefore, standard accuracy is uninformative, and we require metrics suited for rare event classification.

## 2. Evaluation Metrics for Rare Events

### Area Under the ROC Curve (AUC-ROC)

Machine learning models output continuous probabilities rather than direct binary labels (e.g., customer A has a 78% probability of repurchasing, while customer B has 4%).

The **AUC-ROC** evaluates how well the algorithm ranks customers across all potential decision thresholds:

* **Intuitive interpretation**: If we randomly pick one customer who repurchased (Class 1) and one who did not (Class 0), what is the probability that the model assigns a higher probability to the true buyer?
* **AUC = 0.50**: Performance equivalent to random chance (coin toss).
* **AUC = 1.00**: Perfect model; all true buyers are ranked above non-buyers.
* **AUC between 0.60 and 0.75**: Typical realistic range for e-commerce repeat purchase prediction.

AUC-ROC provides an aggregate performance measure independent of any single probability cutoff threshold.

### Precision-Recall AUC (PR-AUC)

When the positive class is very rare (2.3%), AUC-ROC can appear optimistic because the large volume of true negatives inflates specificity.

The **Precision-Recall Curve (PR-AUC)** focuses exclusively on the minority positive class:

* **Recall**: Out of all customers who actually repurchased, what proportion did the model identify?
* **Precision**: When the model flags a customer as a repeat buyer, what fraction actually repurchased?

**Baseline Benchmarks**:
* For **AUC-ROC**, the random baseline is always **0.5000** (50.0%).
* For **PR-AUC**, the random baseline equals the natural prevalence of the positive class: **0.0228** (2.28%).
* A PR-AUC score of 0.12 to 0.15 indicates the model is **5x to 6x more effective than random targeting**.

In [ ]:
# Define feature columns and target label
from pyspark.ml.feature import VectorAssembler

feature_cols = [
    "actual_delivery_days",
    "delay_days",
    "is_delayed",
    "total_items",
    "total_price",
    "shipping_cost",
    "freight_ratio",
    "total_payment",
    "max_payment_installments",
    "total_payment_types",
    "total_reviews",
    "avg_review_score"
]

target_col = "did_purchase_again"

In [ ]:
# Assemble individual feature columns into a single vector column
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

# Transform DataFrame
df_assembled = assembler.transform(df)
df_assembled.select("order_id", "features", target_col).show(3, truncate=False)

### Feature Vector Mapping

The assembled `features` vector aligns with our 12 engineered attributes. For example, for order `3594e05a005ac4d06a72673270ef9ec9`:

| Index | Feature Column | Example Value | Business Interpretation |
| :---: | :--- | :---: | :--- |
| 0 | `actual_delivery_days` | 3.0 | Delivered in 3 days |
| 1 | `delay_days` | -5.0 | Delivered 5 days ahead of estimated date |
| 2 | `is_delayed` | 0.0 | No delivery delay |
| 3 | `total_items` | 1.0 | 1 item purchased |
| 4 | `total_price` | 18.90 | Product subtotal: 18.90 R$ |
| 5 | `shipping_cost` | 8.29 | Shipping fee: 8.29 R$ |
| 6 | `freight_ratio` | 0.3049 | Shipping represents ~30.5% of total cart |
| 7 | `total_payment` | 27.19 | Total payment: 27.19 R$ |
| 8 | `max_payment_installments` | 1.0 | Paid in 1 installment |
| 9 | `total_payment_types` | 1.0 | 1 payment method used |
| 10 | `total_reviews` | 1.0 | 1 review submitted |
| 11 | `avg_review_score` | 4.0 | Rating given: 4 out of 5 stars |

In [ ]:
# Train/Test split: 80% train / 20% test with seed=42 for reproducibility
train_df, test_df = df_assembled.randomSplit([0.8, 0.2], seed=42)

print("Training instances (Train):", train_df.count())
print("Test instances (Test):      ", test_df.count())

In [ ]:
# Baseline Model: Logistic Regression
from pyspark.ml.classification import LogisticRegression

# 1. Model definition with standardization to balance feature scales
lr = LogisticRegression(
    featuresCol="features",
    labelCol="did_purchase_again",
    standardization=True,
    maxIter=20
)

# 2. Train model on the training set
lr_model = lr.fit(train_df)

In [ ]:
# Predictions on the test set
predictions_lr = lr_model.transform(test_df)

# Preview prediction outputs
predictions_lr.select(
    "order_id",
    "did_purchase_again",  # Ground truth
    "rawPrediction",
    "probability",        # [P(no repurchase), P(repurchase)]
    "prediction"          # Binary decision (default threshold 0.5)
).show(5, truncate=False)

### Interpreting Model Probabilities

Spark transforms raw predictions into bounded probabilities between 0 and 1:
`[P(did_purchase_again = 0), P(did_purchase_again = 1)]`

For example, for the first prediction `[0.9814, 0.0186]`:
* The model estimates a 98.14% probability that the customer will not repurchase.
* It estimates a 1.86% probability of repurchase within 90 days.

With a standard 0.50 threshold, all predictions would be rounded down to `0.0`. However, the continuous probabilities provide meaningful relative ranking (e.g., 1.86% vs. 2.14% vs. 1.76%). AUC-ROC and PR-AUC evaluate this continuous ordering rather than the arbitrary 0.50 threshold.

In [ ]:
# Compute evaluation metrics using BinaryClassificationEvaluator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# 1. Evaluator for AUC-ROC (Area Under ROC)
evaluator_roc = BinaryClassificationEvaluator(
    labelCol="did_purchase_again",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# 2. Evaluator for PR-AUC (Area Under Precision-Recall)
evaluator_pr = BinaryClassificationEvaluator(
    labelCol="did_purchase_again",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

auc_roc = evaluator_roc.evaluate(predictions_lr)
auc_pr = evaluator_pr.evaluate(predictions_lr)

print("Logistic Regression - Baseline:")
print(f"AUC-ROC : {auc_roc:.4f}")
print(f"PR-AUC  : {auc_pr:.4f}")

### Baseline Evaluation Results

* **AUC-ROC : 0.6172 (61.72%)**
  * Random guessing baseline is **0.5000** (50%).
  * The baseline model demonstrates discriminative power: in approximately 62% of random positive/negative customer pairs, the model correctly assigns a higher probability to the true repeat buyer.

* **PR-AUC : 0.1162 (11.62%)**
  * Random guessing baseline is the natural positive prevalence: **0.0228** (2.28%).
  * At 0.1162, the baseline model is **5.1x more effective than random targeting**.

## 3. Baseline Explainability: Feature Coefficients

A major advantage of Logistic Regression is interpretability. Each feature has a fitted mathematical weight:

* **Positive coefficient (+)**: Feature increases the log-odds and probability of repeat purchase.
* **Negative coefficient (-)**: Feature decreases the log-odds and probability of repeat purchase.

In [ ]:
import pandas as pd

# Map each feature to its learned coefficient
coefficients = lr_model.coefficients.toArray()
df_weights = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": coefficients
}).sort_values(by="Coefficient", ascending=False)

print("Logistic Regression Coefficients:")
print(df_weights.to_string(index=False))

### Logistic Regression Coefficient Insights

* **`total_reviews` (+4.41) - Top Engagement Signal**: An active customer who leaves reviews is substantially more engaged with the platform and demonstrates a much higher likelihood of repurchasing compared to a passive buyer.
* **`total_items` (+0.22) - Initial Cart Depth**: Customers who purchase multiple items on their first order exhibit higher repeat purchase propensity.
* **`freight_ratio` (+0.16)**: Moderate positive correlation, reflecting regional or bundled order dynamics.
* **`actual_delivery_days` (-0.0075) and `delay_days` (-0.0015) - Delivery Friction**: Negative coefficients confirm that longer transit times and delivery delays reduce repeat purchase probability. Fast, reliable delivery directly drives customer retention.
* **`total_payment` (-0.0039) - Big-Ticket Order Effect**: Higher initial expenditures slightly reduce 90-day repeat purchase probability, consistent with durable goods or furniture purchases that do not require near-term replenishment.

In [ ]:
# Non-Linear Model: Random Forest Classifier
from pyspark.ml.classification import RandomForestClassifier

# 1. Model definition (50 trees, max depth 6, seed 42)
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="did_purchase_again",
    numTrees=50,
    maxDepth=6,
    seed=42
)

# 2. Train model on the training set
rf_model = rf.fit(train_df)
print("Random Forest model trained successfully.")

# 3. Predict on the test set
predictions_rf = rf_model.transform(test_df)

# 4. Evaluate performance
auc_roc_rf = evaluator_roc.evaluate(predictions_rf)
auc_pr_rf = evaluator_pr.evaluate(predictions_rf)

print("Random Forest Evaluation:")
print(f"AUC-ROC : {auc_roc_rf:.4f} (vs {auc_roc:.4f} for Baseline)")
print(f"PR-AUC  : {auc_pr_rf:.4f} (vs {auc_pr:.4f} for Baseline)")

### Random Forest vs. Baseline Comparison

Random Forest outperforms the linear baseline across both metrics:

* **AUC-ROC**: Increases from **0.6172** to **0.6295** (+1.23 points of discriminative power).
* **PR-AUC**: Increases from **0.1162** to **0.1312** (+1.50 points). Compared to the random baseline of 0.0228 (2.28%), Random Forest is **5.8x more effective than random guessing** at identifying repeat buyers.

In [ ]:
# Extract Random Forest Gini Feature Importances
importances = rf_model.featureImportances.toArray()

df_rf_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("Random Forest Feature Importances:")
print(df_rf_importance.to_string(index=False))

## 4. Random Forest Feature Importance Analysis

* **Customer Engagement Dominance (`total_reviews`: ~77.9%)**: Engagement is by far the most decisive signal in tree splits. A customer who actively submits reviews exhibits strong platform interaction, distinguishing future repeat buyers from one-time shoppers.
* **Logistics as Primary Secondary Drivers (`delay_days` and `actual_delivery_days`: ~6.0% combined)**: Delivery speed and schedule adherence represent the second most impactful factor, confirming that logistics directly impacts customer retention.
* **Financial Attributes Form a Balanced Tier (~2.2% to 2.7% each)**: `shipping_cost`, `total_payment`, `max_payment_installments`, `total_price`, and `freight_ratio` contribute comparable split gains throughout the ensemble.
* **Binary Flag Redundancy (`is_delayed`: ~0.05%)**: The binary indicator has low importance because `delay_days` is continuous and already preserves full granular delay information.